# Comparación de estrategias de ecualización

En este cuaderno vas a comparar varias formas de mejorar contraste sobre una misma imagen de trabajo. El objetivo no es memorizar funciones, sino aprender a mirar qué cambia con cada estrategia.


## Objetivo

Comparar cuatro estrategias de mejora de contraste y sostener una decisión simple a partir de observación visual, histogramas y una medida numérica básica.

## Resultados de aprendizaje

Al finalizar este cuaderno, vas a poder:

- distinguir ecualización en grises, por canal, sobre `V` y con `CLAHE`;
- explicar ventajas y límites de cada estrategia;
- interpretar histogramas sin separarlos del resultado visual;
- elegir una estrategia razonable según el problema.

## Relación con la secuencia

Este cuaderno continúa `003 - mejora de imagen y ecualización básica`. Ahí viste transformaciones simples. Acá el foco está en comparar métodos parecidos y aprender a no tratarlos como equivalentes.


## Módulos que vamos a usar

- `cv2`: para ecualizar, convertir espacios de color y aplicar `CLAHE`.
- `numpy`: para calcular una medida simple de dispersión.
- `matplotlib.pyplot`: para comparar imágenes e histogramas.
- `pathlib.Path`: para abrir la imagen de trabajo.


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

ruta_imagen = Path("Imagenes") / "valeria.png"
imagen_bgr = cv2.imread(str(ruta_imagen), cv2.IMREAD_COLOR)
if imagen_bgr is None:
    raise FileNotFoundError(f"No se pudo leer la imagen: {ruta_imagen}")

def construir_version_apagada(imagen_bgr):
    imagen_menos_contraste = cv2.convertScaleAbs(imagen_bgr, alpha=0.58, beta=-18)
    velo_gris = np.full_like(imagen_menos_contraste, 28)
    imagen_menos_contraste = cv2.addWeighted(imagen_menos_contraste, 0.88, velo_gris, 0.12, 0)
    imagen_menos_contraste = cv2.GaussianBlur(imagen_menos_contraste, (3, 3), 0)
    return imagen_menos_contraste

imagen_apagada_bgr = construir_version_apagada(imagen_bgr)
imagen_apagada_rgb = cv2.cvtColor(imagen_apagada_bgr, cv2.COLOR_BGR2RGB)


## 1. Punto de partida

Antes de comparar métodos, conviene fijar un problema común. Vamos a trabajar sobre una versión más apagada de la imagen para que las diferencias entre estrategias sean más visibles.


In [ ]:
plt.figure(figsize=(8, 6), constrained_layout=True)
plt.imshow(imagen_apagada_rgb)
plt.title("Imagen base de trabajo", fontweight="bold", loc="left")
plt.axis("off")
plt.show()


In [ ]:
imagen_apagada_gris = cv2.cvtColor(imagen_apagada_bgr, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(9, 4), constrained_layout=True)
plt.hist(imagen_apagada_gris.ravel(), bins=256, color="gray")
plt.title("Histograma de la imagen base", fontweight="bold", loc="left")
plt.xlabel("Intensidad")
plt.ylabel("Cantidad de píxeles")
plt.grid(alpha=0.3)
plt.show()


Fijate que las intensidades están bastante concentradas. Esa compresión del histograma sugiere bajo contraste, pero no alcanza con mirar la curva: hay que ver qué hace cada método sobre la imagen real.


## 2. Cuatro estrategias

Vamos a construir cuatro salidas:

1. ecualización global en escala de grises;
2. ecualización independiente por canal BGR;
3. ecualización del canal `V` en HSV;
4. `CLAHE` como estrategia adaptativa en grises.


In [ ]:
# Estrategia 1: ecualización global en escala de grises.
ecualizada_gris = cv2.equalizeHist(imagen_apagada_gris)

# Estrategia 2: ecualización por canal BGR.
canal_b, canal_g, canal_r = cv2.split(imagen_apagada_bgr)
b_eq = cv2.equalizeHist(canal_b)
g_eq = cv2.equalizeHist(canal_g)
r_eq = cv2.equalizeHist(canal_r)
ecualizada_bgr = cv2.merge([b_eq, g_eq, r_eq])
ecualizada_bgr_rgb = cv2.cvtColor(ecualizada_bgr, cv2.COLOR_BGR2RGB)

# Estrategia 3: ecualización del canal V en HSV.
imagen_hsv = cv2.cvtColor(imagen_apagada_bgr, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(imagen_hsv)
v_eq = cv2.equalizeHist(v)
ecualizada_hsv = cv2.merge([h, s, v_eq])
ecualizada_hsv_rgb = cv2.cvtColor(ecualizada_hsv, cv2.COLOR_HSV2RGB)

# Estrategia 4: CLAHE en grises.
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
ecualizada_clahe = clahe.apply(imagen_apagada_gris)


## 3. Primera comparación global

Ahora vamos a mirar las cuatro salidas juntas. La comparación inicial sirve para detectar qué método parece más agresivo, cuál conserva mejor los colores y cuál mejora mejor el contraste local.


In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

ejes[0, 0].imshow(imagen_apagada_rgb)
ejes[0, 0].set_title("Base de trabajo", fontweight="bold", loc="left")
ejes[0, 0].axis("off")

ejes[0, 1].imshow(ecualizada_gris, cmap="gray")
ejes[0, 1].set_title("Ecualización en grises", fontweight="bold", loc="left")
ejes[0, 1].axis("off")

ejes[0, 2].imshow(ecualizada_bgr_rgb)
ejes[0, 2].set_title("Ecualización por canal BGR", fontweight="bold", loc="left")
ejes[0, 2].axis("off")

ejes[1, 0].imshow(ecualizada_hsv_rgb)
ejes[1, 0].set_title("Ecualización sobre V", fontweight="bold", loc="left")
ejes[1, 0].axis("off")

ejes[1, 1].imshow(ecualizada_clahe, cmap="gray")
ejes[1, 1].set_title("CLAHE en grises", fontweight="bold", loc="left")
ejes[1, 1].axis("off")

ejes[1, 2].axis("off")
plt.show()


No mires solo cuál “se ve más fuerte”. Mirá también si aparecen colores extraños, zonas demasiado duras o detalles exagerados. Una mejora útil no es solo la que aumenta el contraste, sino la que lo hace de un modo pertinente para la tarea.


## 4. Comparar un recorte local

Las diferencias finas se ven mejor en un recorte pequeño. Eso ayuda a no quedarse solo con la impresión general.


In [ ]:
fila_inicial, fila_final = 110, 280
columna_inicial, columna_final = 110, 280

recortes = [
    imagen_apagada_rgb[fila_inicial:fila_final, columna_inicial:columna_final],
    ecualizada_bgr_rgb[fila_inicial:fila_final, columna_inicial:columna_final],
    ecualizada_hsv_rgb[fila_inicial:fila_final, columna_inicial:columna_final],
]

titulos = ["Base", "Por canal BGR", "Sobre V en HSV"]

fig, ejes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for eje, imagen, titulo in zip(ejes, recortes, titulos):
    eje.imshow(imagen)
    eje.set_title(titulo, fontweight="bold", loc="left")
    eje.axis("off")
plt.show()


En esta comparación conviene mirar especialmente si cambian los tonos de piel, si aparecen dominantes de color o si el detalle se vuelve artificial. En imágenes color, ese tipo de distorsión importa mucho.


## 5. Una medida simple para acompañar la observación

Vamos a usar el desvío estándar de intensidades en grises como una referencia mínima. No reemplaza la observación, pero puede ayudar a ver si una imagen quedó más extendida en términos tonales.


In [ ]:
def desvio_estandar_gris(imagen):
    return float(np.std(imagen))

print(f"Base de trabajo: {desvio_estandar_gris(imagen_apagada_gris):.2f}")
print(f"Ecualización global: {desvio_estandar_gris(ecualizada_gris):.2f}")
print(f"CLAHE: {desvio_estandar_gris(ecualizada_clahe):.2f}")


Esta medida puede acompañar la lectura, pero no decide sola. Una imagen puede tener más dispersión tonal y aun así ser peor si distorsiona color o exagera ruido. Por eso conviene usar números y observación juntos.


## Limitaciones y criterio de decisión

En esta comparación no hay un método universalmente mejor. En general:

- la ecualización por canal puede alterar bastante el color;
- trabajar sobre `V` suele conservar mejor la apariencia cromática;
- `CLAHE` puede recuperar contraste local sin uniformar toda la imagen de la misma manera;
- la ecualización global en grises sirve para entender el mecanismo, pero pierde información de color.


## Actividad breve

Elegí dos de las cuatro estrategias y escribí una comparación corta donde respondas:

1. ¿qué mejora concreta produce cada una?
2. ¿qué problema introduce o podría introducir?
3. ¿cuál usarías si tu objetivo fuera segmentar por color en un paso posterior?


## Cierre

Comparar estrategias de ecualización no consiste en buscar la imagen “más impactante”, sino en elegir la transformación que mejor preserve la información relevante para la tarea siguiente. Esa mirada crítica es más importante que aprender una función aislada.
